# Webinar #4 — Logical Agents & Introduction to Prolog

**Knowledge representation with propositional logic, and a hands-on intro to Prolog.**

This notebook is *fully solved* and uses only the Python standard library (`itertools`),
so it runs anywhere with no installs. It mirrors the webinar deck:

1. Propositional-logic toolkit — build formulas, truth tables, classify validity
2. Formalizing English sentences (the 3 deck examples)
3. Normal forms — a CNF converter (the 4 textbook steps)
4. Textbook problems — AIMA Ch. 7 validity & entailment checks
5. Prolog — a ~40-line backward-chaining engine (unification + backtracking)
6. Exercises (with solutions)

> See `README.md` in this folder for the theory behind every section.

---
## Part 1 — A tiny propositional-logic toolkit

We represent a formula as a small tree (an **AST**). Each node knows how to
`eval` itself against a *model* (a dict mapping each atom to `True`/`False`)
and which `symbols` it contains. That's all we need to build truth tables.

Connectives: `Not`, `And`, `Or`, `Implies`, `Iff`. Remember the trap —
`A -> B` is **false only** when `A` is true and `B` is false.

In [ ]:
from itertools import product

class Formula:
    """Base class. The dunder methods let us write formulas naturally:
    ~A, A & B, A | B  (and .implies / .iff for the arrow connectives)."""
    def __and__(self, o):  return And(self, o)
    def __or__(self, o):   return Or(self, o)
    def __invert__(self):  return Not(self)
    def implies(self, o):  return Implies(self, o)
    def iff(self, o):      return Iff(self, o)

class Var(Formula):
    def __init__(self, name): self.name = name
    def eval(self, m): return m[self.name]
    def symbols(self): return {self.name}
    def __repr__(self): return self.name

class Not(Formula):
    def __init__(self, x): self.x = x
    def eval(self, m): return not self.x.eval(m)
    def symbols(self): return self.x.symbols()
    def __repr__(self): return f"~{self.x!r}"

class And(Formula):
    def __init__(self, a, b): self.a, self.b = a, b
    def eval(self, m): return self.a.eval(m) and self.b.eval(m)
    def symbols(self): return self.a.symbols() | self.b.symbols()
    def __repr__(self): return f"({self.a!r} & {self.b!r})"

class Or(Formula):
    def __init__(self, a, b): self.a, self.b = a, b
    def eval(self, m): return self.a.eval(m) or self.b.eval(m)
    def symbols(self): return self.a.symbols() | self.b.symbols()
    def __repr__(self): return f"({self.a!r} | {self.b!r})"

class Implies(Formula):
    def __init__(self, a, b): self.a, self.b = a, b
    def eval(self, m): return (not self.a.eval(m)) or self.b.eval(m)
    def symbols(self): return self.a.symbols() | self.b.symbols()
    def __repr__(self): return f"({self.a!r} => {self.b!r})"

class Iff(Formula):
    def __init__(self, a, b): self.a, self.b = a, b
    def eval(self, m): return self.a.eval(m) == self.b.eval(m)
    def symbols(self): return self.a.symbols() | self.b.symbols()
    def __repr__(self): return f"({self.a!r} <=> {self.b!r})"

Now the reasoning helpers: enumerate **all models**, print a **truth table**,
**classify** a formula (valid / unsatisfiable / neither), and test **entailment**.

In [ ]:
def all_models(symbols):
    """Yield every True/False assignment over the given atoms."""
    symbols = sorted(symbols)
    for combo in product([False, True], repeat=len(symbols)):
        yield dict(zip(symbols, combo))

def print_truth_table(f, label='formula'):
    syms = sorted(f.symbols())
    header = syms + [label]
    widths = [max(len(h), 5) for h in header]
    def row(vals):
        return ' | '.join(str(v)[:1].ljust(w) if isinstance(v, bool) else str(v).ljust(w)
                          for v, w in zip(vals, widths))
    print(row(header))
    print('-' * (sum(widths) + 3 * len(widths)))
    for m in all_models(syms):
        print(row([m[s] for s in syms] + [f.eval(m)]))

def classify(f):
    results = [f.eval(m) for m in all_models(f.symbols())]
    if all(results):     return 'VALID (tautology)'
    if not any(results): return 'UNSATISFIABLE (contradiction)'
    return 'NEITHER (satisfiable, contingent)'

def entails(premises, conclusion):
    """KB |= alpha  iff  (KB -> alpha) is valid."""
    kb = premises[0]
    for p in premises[1:]:
        kb = And(kb, p)
    return classify(Implies(kb, conclusion)).startswith('VALID')

# quick smoke test
P, Q = Var('P'), Var('Q')
print_truth_table(Implies(P, Q), 'P=>Q')
print()
print('P => P  is', classify(Implies(P, P)))

---
## Part 2 — Formalizing English sentences

The three deck examples. Reminders: **"unless" = "if not"**, and
**"but not both" = XOR**.

- **Ex 1:** *If the alarm is set, then the house is secure, unless the door is unlocked.*
  → `(A & ~D) => S`
- **Ex 2:** *Either the server is down or the network is slow, but not both.* (XOR)
  → `(S | N) & ~(S & N)`
- **Ex 3:** *If it rains or snows, then the match is cancelled unless the stadium is covered.*
  → `((R | Sn) & ~D) => C`

In [ ]:
# Example 1
A, S, D = Var('A'), Var('S'), Var('D')
ex1 = Implies(And(A, Not(D)), S)
print('Ex1:', ex1, '->', classify(ex1))

# Example 2 (exclusive or)
Sd, N = Var('S'), Var('N')
ex2 = And(Or(Sd, N), Not(And(Sd, N)))
print('Ex2:', ex2, '->', classify(ex2))
print_truth_table(ex2, 'XOR')

In [ ]:
# Example 3
R, Sn, C, Dc = Var('R'), Var('Sn'), Var('C'), Var('D')
ex3 = Implies(And(Or(R, Sn), Not(Dc)), C)
print('Ex3:', ex3, '->', classify(ex3))
print_truth_table(ex3, 'match?')

---
## Part 3 — Conjunctive Normal Form (CNF)

CNF = an **AND of clauses**, each clause an **OR of literals**. We implement the
four textbook steps as separate functions, then chain them in `to_cnf`:

1. `elim_iff` — replace `A <=> B` with `(A=>B) & (B=>A)`
2. `elim_implies` — replace `A => B` with `~A | B`
3. `push_neg` — De Morgan + drop double negation
4. `distribute` — push `OR` inside `AND`

In [ ]:
def elim_iff(f):
    if isinstance(f, Iff):
        a, b = elim_iff(f.a), elim_iff(f.b)
        return And(Implies(a, b), Implies(b, a))
    if isinstance(f, Implies): return Implies(elim_iff(f.a), elim_iff(f.b))
    if isinstance(f, And):     return And(elim_iff(f.a), elim_iff(f.b))
    if isinstance(f, Or):      return Or(elim_iff(f.a), elim_iff(f.b))
    if isinstance(f, Not):     return Not(elim_iff(f.x))
    return f

def elim_implies(f):
    if isinstance(f, Implies):
        return Or(Not(elim_implies(f.a)), elim_implies(f.b))
    if isinstance(f, And): return And(elim_implies(f.a), elim_implies(f.b))
    if isinstance(f, Or):  return Or(elim_implies(f.a), elim_implies(f.b))
    if isinstance(f, Not): return Not(elim_implies(f.x))
    return f

def push_neg(f):
    if isinstance(f, Not):
        g = f.x
        if isinstance(g, Not): return push_neg(g.x)                              # ~~A -> A
        if isinstance(g, And): return Or(push_neg(Not(g.a)), push_neg(Not(g.b))) # De Morgan
        if isinstance(g, Or):  return And(push_neg(Not(g.a)), push_neg(Not(g.b)))# De Morgan
        return f
    if isinstance(f, And): return And(push_neg(f.a), push_neg(f.b))
    if isinstance(f, Or):  return Or(push_neg(f.a), push_neg(f.b))
    return f

def distribute(f):
    if isinstance(f, And): return And(distribute(f.a), distribute(f.b))
    if isinstance(f, Or):
        a, b = distribute(f.a), distribute(f.b)
        if isinstance(a, And): return And(distribute(Or(a.a, b)), distribute(Or(a.b, b)))
        if isinstance(b, And): return And(distribute(Or(a, b.a)), distribute(Or(a, b.b)))
        return Or(a, b)
    return f

def to_cnf(f):
    return distribute(push_neg(elim_implies(elim_iff(f))))

In [ ]:
# Worked examples from the deck
B, Cc = Var('B'), Var('C')
f1 = Implies(A, And(B, Cc))          # A => (B & C)
print('CNF of', f1, '=', to_cnf(f1))

f2 = Implies(And(A, B), Cc)          # (A & B) => C  (already CNF once arrows removed)
print('CNF of', f2, '=', to_cnf(f2))

# Sanity check: a formula and its CNF must be logically equivalent
same = all(f1.eval(m) == to_cnf(f1).eval(m) for m in all_models(f1.symbols()))
print('CNF equivalent to original?', same)

---
## Part 4 — Textbook problems (AIMA, Chapter 7)

Exercise 7.10: decide **valid** vs **neither** for each sentence. We just let
`classify` build the truth table and read off the verdict.

In [ ]:
Smoke, Fire, Heat = Var('Smoke'), Var('Fire'), Var('Heat')
Big, Dumb = Var('Big'), Var('Dumb')

problems = {
    'a) Smoke => Smoke':
        Implies(Smoke, Smoke),
    'b) Smoke => Fire':
        Implies(Smoke, Fire),
    'c) (Smoke=>Fire) => (~Smoke=>~Fire)':
        Implies(Implies(Smoke, Fire), Implies(Not(Smoke), Not(Fire))),
    'd) Smoke | Fire | ~Fire':
        Or(Or(Smoke, Fire), Not(Fire)),
    'e) ((Smoke&Heat)=>Fire) <=> ((Smoke=>Fire)|(Heat=>Fire))':
        Iff(Implies(And(Smoke, Heat), Fire),
            Or(Implies(Smoke, Fire), Implies(Heat, Fire))),
    'f) Big | Dumb | (Big=>Dumb)':
        Or(Or(Big, Dumb), Implies(Big, Dumb)),
    'g) (Big & Dumb) | ~Dumb':
        Or(And(Big, Dumb), Not(Dumb)),
}

for name, formula in problems.items():
    print(f'{name:52s} -> {classify(formula)}')

### Entailment problem

Does `{F => P, D => P}` entail `(F & D) => P`? (Deck slide 21.) Test whether
`(KB -> conclusion)` is valid.

In [ ]:
F, P, Dd = Var('F'), Var('P'), Var('D')
kb = [Implies(F, P), Implies(Dd, P)]
goal = Implies(And(F, Dd), P)
print('KB =', kb)
print('Does KB entail (F & D) => P ?', entails(kb, goal))

---
## Part 5 — Introduction to Prolog

Prolog is **declarative**: you write **facts** and **rules**, then ask **queries**;
the engine searches for answers by **unification** (making two terms identical) and
**backtracking** (undoing choices that lead to dead ends).

Here are the deck's programs in real SWI-Prolog syntax (install from
<https://www.swi-prolog.org/> to run them):

```prolog
parent(john, mary).
parent(mary, sam).
grandparent(X, Y) :- parent(X, Z), parent(Z, Y).
?- grandparent(john, sam).   % true.

fever(john).  cough(john).
covid(X) :- fever(X), cough(X).
?- covid(john).              % true.
```

Below we build a **mini Prolog engine in pure Python** so you can watch the same
search run without leaving the notebook.

In [ ]:
# --- Terms and logic variables ---
class Term:
    """A Prolog term: a functor with 0+ argument terms, e.g. parent(john, mary)."""
    def __init__(self, functor, args=()):
        self.functor = functor
        self.args = list(args)
    def __repr__(self):
        if not self.args: return self.functor
        return f"{self.functor}({', '.join(map(repr, self.args))})"

def V(name):   return ('var', name)   # a logic variable is just a tagged tuple
def is_var(t): return isinstance(t, tuple) and t[0] == 'var'

def walk(t, subst):
    """Follow variable bindings until we reach a non-variable (or an unbound var)."""
    while is_var(t) and t in subst:
        t = subst[t]
    return t

def unify(x, y, subst):
    """Try to make x and y identical, extending the substitution. None = fail."""
    if subst is None: return None
    x, y = walk(x, subst), walk(y, subst)
    if is_var(x):
        s = dict(subst); s[x] = y; return s
    if is_var(y):
        s = dict(subst); s[y] = x; return s
    if isinstance(x, Term) and isinstance(y, Term):
        if x.functor != y.functor or len(x.args) != len(y.args): return None
        for a, b in zip(x.args, y.args):
            subst = unify(a, b, subst)
            if subst is None: return None
        return subst
    return subst if x == y else None

In [ ]:
# --- The knowledge base + backward-chaining solver ---
_counter = [0]
def rename_rule(head, body):
    """Give a whole rule fresh variables (head + body share them) so two uses
    of the same rule don't clash."""
    _counter[0] += 1
    tag = _counter[0]
    def rec(t):
        if is_var(t): return ('var', f'{t[1]}#{tag}')
        if isinstance(t, Term): return Term(t.functor, [rec(a) for a in t.args])
        return t
    return rec(head), [rec(b) for b in body]

class KB:
    def __init__(self): self.rules = []          # each rule = (head_Term, [body_Terms])
    def fact(self, head):       self.rules.append((head, []))
    def rule(self, head, body): self.rules.append((head, list(body)))

    def solve(self, goals, subst):
        if not goals:                 # nothing left to prove -> success
            yield subst
            return
        goal, rest = goals[0], goals[1:]
        for head, body in self.rules:
            head, body = rename_rule(head, body)
            s = unify(goal, head, dict(subst))     # try to match this clause
            if s is not None:
                yield from self.solve(list(body) + rest, s)   # prove body, then the rest

    def query(self, *goals):
        return list(self.solve(list(goals), {}))

In [ ]:
# Program 1: family relationships
kb = KB()
kb.fact(Term('parent', [Term('john'), Term('mary')]))
kb.fact(Term('parent', [Term('mary'), Term('sam')]))
kb.rule(Term('grandparent', [V('X'), V('Y')]),
        [Term('parent', [V('X'), V('Z')]), Term('parent', [V('Z'), V('Y')])])

print('grandparent(john, sam)? ->', bool(kb.query(Term('grandparent', [Term('john'), Term('sam')]))))

# Variable query: who is a grandparent of whom?
for s in kb.query(Term('grandparent', [V('X'), V('Y')])):
    print('  X =', walk(V('X'), s), ', Y =', walk(V('Y'), s))

In [ ]:
# Program 2: tiny expert system
med = KB()
med.fact(Term('fever', [Term('john')]))
med.fact(Term('cough', [Term('john')]))
med.rule(Term('covid', [V('X')]), [Term('fever', [V('X')]), Term('cough', [V('X')])])
print('covid(john)? ->', bool(med.query(Term('covid', [Term('john')]))))

# Program 3: mortal philosophers
phil = KB()
phil.rule(Term('mortal', [V('X')]), [Term('human', [V('X')])])
phil.fact(Term('human', [Term('socrates')]))
print('mortal(socrates)? ->', bool(phil.query(Term('mortal', [Term('socrates')]))))

**How the engine answered `mortal(socrates)`** — exactly the deck's goal-execution steps:

1. Goal `mortal(socrates)` unifies with rule head `mortal(X)` → binds `X = socrates`.
2. Goal is replaced by the rule body `human(X)` → `human(socrates)`.
3. `human(socrates)` matches a fact → success.
4. No goals left → answer **true**.

---
## Part 6 — Exercises (with solutions)

**Ex 1 — Textbook.** Confirm a couple of the AIMA 7.10 verdicts by re-deriving them.
**Ex 2 — Prolog.** Add a `sibling/2` rule to the family KB.
**Ex 3 (bonus) — Wumpus World** rule sketch.

In [ ]:
# Ex 1 solution: re-check two sentences and show WHY (a countermodel for 'neither')
g = Or(And(Big, Dumb), Not(Dumb))   # (g) -> Neither
print('(g)', g, '->', classify(g))
for m in all_models(g.symbols()):
    if not g.eval(m):
        print('  falsified by', m, '=> not valid; but it IS true for other models => neither')
        break

In [ ]:
# Ex 2 solution: sibling(X, Y) :- parent(Z, X), parent(Z, Y), X \= Y.
# (our mini engine has no inequality, so we add more parents and filter X != Y in Python)
fam = KB()
for p, c in [('john','mary'), ('john','peter'), ('mary','sam'), ('mary','anna')]:
    fam.fact(Term('parent', [Term(p), Term(c)]))
fam.rule(Term('sibling', [V('X'), V('Y')]),
         [Term('parent', [V('Z'), V('X')]), Term('parent', [V('Z'), V('Y')])])

pairs = set()
for s in fam.query(Term('sibling', [V('X'), V('Y')])):
    x, y = walk(V('X'), s).functor, walk(V('Y'), s).functor
    if x != y:
        pairs.add(tuple(sorted((x, y))))
print('sibling pairs:', sorted(pairs))

**Ex 3 (bonus) — Wumpus World in Prolog (sketch).** A cell is *safe* if it has no pit and
no wumpus. "No breeze in a cell" implies its neighbours have no pit; "no stench" implies its
neighbours have no wumpus.

```prolog
adjacent(X, Y) :- ...            % define the grid neighbours

safe(C) :- \+ pit(C), \+ wumpus(C).

% If a visited cell has no breeze, none of its neighbours is a pit:
no_pit(N) :- adjacent(C, N), visited(C), \+ breeze(C).
no_wumpus(N) :- adjacent(C, N), visited(C), \+ stench(C).
```

Install SWI-Prolog and flesh this out to earn the goodie in the deck!

---
### Recap

- Propositional logic = true/false statements + connectives; **truth tables** settle everything.
- **CNF** puts any formula into AND-of-ORs form via 4 mechanical steps.
- **Valid / satisfiable / unsatisfiable** and **entailment** all reduce to checking models.
- **Prolog** = facts + rules + queries, answered by **unification + backtracking** (SLD resolution).

Next: download SWI-Prolog and run these programs for real.